# The check on markets never looked at

Plan committed before these markets were downloaded: see [PLAN_more_inputs.md](PLAN_more_inputs.md) (*Clean checks*).

- **Markets:** Korea (KOSPI), Taiwan (TAIEX), Hong Kong (Hang Seng) and Japan (Nikkei 225). No choice in this project was made using them.
- **Same code and settings as step 9:** the four sets of inputs, seven models, test years 2017–2026, the same days for every set. Run once.
- **Expected:** the same pattern as step 9: no gain from B or C, and a close-to-close gain from D.

`more_inputs.py new_markets` runs the models. This notebook scores them with step 9's code.

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics import roc_auc_score

from classifiers import overview
from data import NEW_MARKETS, load_all
from evaluation import bootstrap_range, holm, mcnemar
from models import pooled
from more_inputs import MODELS, RUNS, SAVED, run_all

markets = load_all(include_holdout=True, markets=NEW_MARKETS)
preds = run_all(markets, folder=SAVED / "new_markets")  # loads the saved runs

def together(target, s, model):
    """One model's predictions on every market, one row per market-day."""
    return pd.concat([preds[target, s, model, m].assign(market=m) for m in markets])

def right(p):
    return (p["p_up"] > 0.5) == p["up"]

pd.Series({m: len(preds["close to close", "A", "LDA", m]) for m in markets}, name="test days").to_frame().T

## Overview

Mean AUC over the 40 folds (4 markets × 10 test years), per model and set. A coin flip scores 0.50.

In [ ]:
pd.DataFrame({f"{s}, {target}": overview({model: {m: preds[target, s, model, m] for m in markets} for model in MODELS})["mean AUC"]
              for target, sets in RUNS.items() for s in sets}).round(3)

## Main result: better than A?

Each model with set B, C or D against the same model with set A, on the same days (all markets and years together). McNemar's test counts days where only one of the two was right. Holm adjusts over the 21 close-to-close comparisons. The AUC range is from whole weeks drawn at random.

In [ ]:
def against_a(target, s, model):
    a, b = together(target, "A", model), together(target, s, model)
    assert a.index.equals(b.index) and (a["market"] == b["market"]).all()  # the same days
    only_b, only_a, p = mcnemar(b["up"], b["p_up"] > 0.5, a["p_up"] > 0.5)
    daily = pd.DataFrame({"up": b["up"], "a": a["p_up"], "b": b["p_up"]})
    auc_gain = lambda f: roc_auc_score(f["up"], f["b"]) - roc_auc_score(f["up"], f["a"])
    low, high = bootstrap_range(daily, auc_gain)
    return {"accuracy gain, points": 100 * (right(b).mean() - right(a).mean()), "AUC gain": auc_gain(daily),
            "AUC gain, low": low, "AUC gain, high": high,
            "days only it was right": only_b, "days only A was right": only_a, "p": p}

main = pd.DataFrame({(s, model): against_a("close to close", s, model) for s in "BCD" for model in MODELS}).T
main["p (Holm)"] = holm(main["p"])
main.round(3)

Open to close (buy at the open, sell at the close): set D against A, Holm over these 7.

In [ ]:
tradeable = pd.DataFrame({model: against_a("open to close", "D", model) for model in MODELS}).T
tradeable["p (Holm)"] = holm(tradeable["p"])
tradeable.round(3)

## Any edge at all?

Pooled accuracy minus the fair bar, per model and market. Holm adjusts within each set (28 tests), as in step 8.

In [ ]:
edge = pd.DataFrame([{"target": target, "set": s, "model": model, "market": m,
                      "gain": pooled(preds[target, s, model, m])["accuracy %"] - pooled(preds[target, s, model, m])["fair bar %"],
                      "p": pooled(preds[target, s, model, m])["p vs fair"]}
                     for target, sets in RUNS.items() for s in sets for model in MODELS for m in markets])
edge["p (Holm)"] = edge.groupby(["target", "set"])["p"].transform(holm)
edge.groupby(["target", "set"]).agg(
    tests=("p", "size"), mean_gain=("gain", "mean"), above=("gain", lambda g: (g > 0).sum()),
    significant=("p", lambda p: (p < 0.05).sum()), significant_after_holm=("p (Holm)", lambda p: (p < 0.05).sum())).round(2)

Each set's gains over the fair bar, in points (model × market).

In [ ]:
for (target, s), table in edge.groupby(["target", "set"]):
    print(f"set {s}, {target}")
    display(table.pivot(index="model", columns="market", values="gain").round(1))

## 2017–2022 against 2023–2026

Accuracy gain over A in each period, in points, averaged over the seven models.

In [ ]:
periods = {"2017-2022": range(2017, 2023), "2023-2026": range(2023, 2027)}
rows = {}
for target, sets in RUNS.items():
    for s in sets:
        if s == "A":
            continue
        for period, years in periods.items():
            gains = []
            for model in MODELS:
                a, b = together(target, "A", model), together(target, s, model)
                keep = a["year"].isin(years)
                gains.append(100 * (right(b)[keep].mean() - right(a)[keep].mean()))
            rows[f"{s}, {target}", period] = np.mean(gains)
pd.Series(rows).unstack().round(2)

## What tsfresh kept

For each market, the 11 training cutoffs (2015–2025): how often no feature passed (so the top 10 were used), and how many passed otherwise.

In [ ]:
rows, kept = {}, Counter()
for m in markets:
    chosen = pd.read_pickle(SAVED / f"tsfresh_selection_close_to_close_{m}.pkl")
    passed = [len(cols) for cols, fell_back in chosen.values() if not fell_back]
    rows[m] = {"cutoffs": len(chosen), "fell back to the top 10": sum(f for _, f in chosen.values()),
               "features passing, median": np.median(passed) if passed else np.nan,
               "features passing, most": max(passed) if passed else np.nan}
    kept.update(c for cols, _ in chosen.values() for c in cols)
display(pd.DataFrame(rows).T)
pd.Series(dict(kept.most_common(10)), name="times kept").to_frame()

CONCLUSIONS